# 05 Secrets and Secret Scopes

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Keep credentials out of code: create a <b>secret scope</b>, store secrets in it, read them in notebooks with <code>dbutils.secrets.get</code>, see how redaction works, control who can read them, and know the alternatives (Azure Key Vault-backed scopes, Unity Catalog connections and service credentials).
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
A database password or API token pasted into a notebook ends up in Git history, logs, screenshots and exports, and once leaked it has to be rotated everywhere. Secret management is a standard security interview question and appears in the exam's governance domain. It's also exactly the kind of habit that keeps a learning project safe on GitHub.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Secret scopes</b>
<ul><li>A <b>secret scope</b> is a named collection of secrets (key → value), with access control lists (ACLs)</li><li><b>Databricks-backed</b> scopes store secrets encrypted by Databricks. Create them with the CLI, the REST API or the SDK</li><li><b>Azure Key Vault-backed</b> scopes read secrets directly from a Key Vault (Azure Databricks), so secrets are managed in Azure (lesson 09_06)</li><li>Notebooks read secrets with <code>dbutils.secrets.get(scope, key)</code>. The value is <b>redacted</b> (<code>[REDACTED]</code>) if you try to display it</li><li>ACL permissions: <code>READ</code>, <code>WRITE</code>, <code>MANAGE</code></li></ul>
</div>

| Option | Best for |
|---|---|
| Secret scope + `dbutils.secrets.get` | Passwords, API tokens, keys used in notebook code |
| Unity Catalog **connections** | External databases (Lakehouse Federation): credentials stored in the connection, governed by UC |
| Unity Catalog **storage / service credentials** | Access to cloud storage and cloud services, without keys in code |
| Environment variables in job/cluster config referencing secrets | `{{secrets/scope/key}}` in Spark config or env vars on classic compute |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team kept a CRM API token in a notebook "temporarily". It was committed to GitHub, a bot found it within hours, and the token had to be revoked, breaking three pipelines that used it. Now the token lives in a secret scope that only the pipeline's service principal and the data engineering group can read. Rotating it means updating one secret, and no code changes.
</div>
<br>
<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Lesson from this repository</b><br>
If you ever kept a password or token in a text file next to your notebooks (even outside the repo), treat it as exposed: revoke or rotate it, and move it to a password manager or a secret scope.
</div>

## 1. Create a secret scope and a secret

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses the Databricks SDK to create a scope called <code>learning</code> and store a demo secret in it.<br><br>
<b>Why it matters</b><br>Secrets are created <b>outside</b> notebooks in practice (CLI, Terraform, CI/CD), so they never appear in code. The SDK does the same as the CLI command <code>databricks secrets create-scope learning</code>. Here the value is generated randomly, so no real credential is ever typed.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Creating scopes uses the Databricks Secrets API. If it's restricted in your workspace, the cell prints the reason, and you can use the CLI from your machine instead: <code>databricks secrets create-scope learning</code> then <code>databricks secrets put-secret learning demo_api_token</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>scope ready: learning</code> and <code>secret stored: demo_api_token</code>.
</div>

In [0]:
import secrets as pysecrets
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
SCOPE, KEY = "learning", "demo_api_token"

try:
    if SCOPE not in [s.name for s in w.secrets.list_scopes()]:
        w.secrets.create_scope(scope=SCOPE)
    print("scope ready:", SCOPE)
    w.secrets.put_secret(scope=SCOPE, key=KEY, string_value="demo-" + pysecrets.token_hex(16))
    print("secret stored:", KEY)
except Exception as e:
    print("could not create the scope or secret here:", type(e).__name__, str(e)[:150])

## 2. List scopes and keys (never values)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the scopes and the keys in the <code>learning</code> scope.<br><br>
<b>Why it matters</b><br>You can see <b>which</b> secrets exist without seeing their values. Listing is safe to share in logs and documentation.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>learning</code> scope, and the key <code>demo_api_token</code> with its last-updated timestamp.
</div>

In [0]:
print("scopes:", [s.name for s in dbutils.secrets.listScopes()])
for meta in dbutils.secrets.list(SCOPE):
    print("key:", meta.key)

## 3. Read a secret and see the redaction

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the secret, tries to print it, and uses it in code.<br><br>
<b>Why it matters</b><br><code>dbutils.secrets.get</code> returns the real value to your code, but notebook output replaces it with <code>[REDACTED]</code>, so it doesn't leak in results, shared notebooks or job logs. Redaction is a safety net, not a security boundary: anyone allowed to run code with <code>READ</code> permission can use the value, which is why ACLs matter.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>[REDACTED]</code> where the value would be printed, then its length and a check that it starts with <code>demo-</code>.
</div>

In [0]:
token = dbutils.secrets.get(scope=SCOPE, key=KEY)
print("token:", token)
print("length:", len(token), "| starts with 'demo-':", token.startswith("demo-"))

## 4. Use a secret where it's needed

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds request headers and JDBC connection options from secrets, as a real pipeline would.<br><br>
<b>Why it matters</b><br>Secrets are passed directly to the libraries that need them (HTTP clients, JDBC readers, SDKs), and never written to files, tables or logs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The header and options dictionaries with the secret values redacted in the output.
</div>

In [0]:
headers = {"Authorization": f"Bearer {dbutils.secrets.get(SCOPE, KEY)}", "Accept": "application/json"}
print(headers)

jdbc_options = {
    "url": "jdbc:postgresql://db.example.com:5432/sales",
    "user": "reporting_user",
    "password": dbutils.secrets.get(SCOPE, KEY),     # in real life: a key such as "pg_password"
    "dbtable": "public.orders",
}
print(jdbc_options)
# spark.read.format("jdbc").options(**jdbc_options).load()   # would connect with the secret

## 5. Control who can read the scope

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the scope's ACLs and shows how to grant read access to a group.<br><br>
<b>Why it matters</b><br>Least privilege: the people and service principals that <b>run</b> a pipeline need <code>READ</code>, while only the owners need <code>MANAGE</code>. The scope creator has <code>MANAGE</code> by default.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
At least one ACL entry: your user with <code>MANAGE</code>. The grant example is shown as code (it needs an existing group).
</div>

In [0]:
try:
    for acl in w.secrets.list_acls(scope=SCOPE):
        print(f"{acl.principal:<40} {acl.permission}")
except Exception as e:
    print("ACLs not available:", type(e).__name__)

# Grant read access to a group (replace with a real group in your workspace):
# from databricks.sdk.service.workspace import AclPermission
# w.secrets.put_acl(scope=SCOPE, principal="data-engineers", permission=AclPermission.READ)

## 6. Secrets in job and cluster configuration

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the <code>{{secrets/scope/key}}</code> reference syntax used in configuration instead of code.<br><br>
<b>Why it matters</b><br>On classic clusters and in some job configurations, you can reference a secret in Spark config or environment variables. The value is resolved at start time and never stored in the configuration.
</div>

```text
Spark config (classic cluster):
  spark.hadoop.fs.azure.account.key.mystorage.dfs.core.windows.net {{secrets/storage/account_key}}

Environment variable (cluster or job compute):
  CRM_API_TOKEN={{secrets/learning/demo_api_token}}
```

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b><br>
For cloud storage, prefer Unity Catalog <b>storage credentials and external locations</b> (no keys at all). For external databases, prefer Unity Catalog <b>connections</b>. Use secret scopes for everything else.
</div>

## 7. Cleanup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deletes the demo secret and scope.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>deleted scope: learning</code> (or a message if it couldn't be deleted).
</div>

In [0]:
try:
    w.secrets.delete_secret(scope=SCOPE, key=KEY)
    w.secrets.delete_scope(scope=SCOPE)
    print("deleted scope:", SCOPE)
except Exception as e:
    print("cleanup skipped:", type(e).__name__)

## Under the hood

```
create-scope / put-secret  (CLI, SDK, Terraform)  ──▶  Databricks secret store (encrypted)
                                                          or Azure Key Vault (KV-backed scope)
notebook: dbutils.secrets.get(scope, key)
   1. check the caller's ACL on the scope (READ)
   2. return the value to the code
   3. output rendering: any printed copy of the value is replaced by [REDACTED]
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> reading a secret is a call to the secrets service, not a Spark job.

In [0]:
# A local stand-in for testing code that needs secrets (never put real values here):
import os
def get_secret(scope, key):
    try:
        return dbutils.secrets.get(scope, key)                      # noqa: F821
    except Exception:
        return os.environ.get(f"{scope.upper()}_{key.upper()}", "test-value")

print("secret available:", bool(get_secret("learning", "demo_api_token")))

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: a credential is in a notebook, a config file or Git history</b><br>
Revoke or rotate it immediately, then move it to a secret scope. Removing it from the latest commit isn't enough, because history keeps it.<br><br>
**⛔ Problem: <code>Secret does not exist with scope: ... and key: ...</code>**<br>A typo in the scope or key, or the secret was created in another workspace.<br><br>
**⛔ Problem: <code>PERMISSION_DENIED</code> reading a secret**<br>The user or service principal running the code lacks <code>READ</code> on the scope.<br><br>
<b>⛔ Problem: secret value written to a table or log by mistake</b><br>Redaction only applies to notebook output. Never write secret values anywhere.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you manage credentials in Databricks?</b><br>
In secret scopes: Databricks-backed, or Azure Key Vault-backed on Azure. Secrets are created through the CLI, API or Terraform, read in code with <code>dbutils.secrets.get</code>, redacted in outputs, and protected by scope ACLs. For storage and external databases I prefer Unity Catalog storage credentials and connections, so no keys appear in code at all.<br><br>

<b>🎤 2. What does redaction do, and is it enough?</b><br>
Printed secret values are replaced with <code>[REDACTED]</code> in notebook output. It prevents accidental exposure, but anyone with <code>READ</code> on the scope can still use the value in code, so ACLs and least privilege are what really protect it.<br><br>

<b>🎤 3. What permissions exist on a secret scope?</b><br>
<code>READ</code> to read secrets, <code>WRITE</code> to add and update them, and <code>MANAGE</code> to also change ACLs and delete the scope.<br><br>

<b>🎤 4. What is a Key Vault-backed secret scope?</b><br>
A scope on Azure Databricks that reads secrets directly from an Azure Key Vault, so secrets are created, rotated and audited in Azure, and Databricks only references them.<br><br>

<b>🎤 5. What do you do if a token was committed to Git?</b><br>
Revoke or rotate it immediately, because it's compromised even if the commit is removed. Then store the new one in a secret scope, and consider scanning the repository and enabling secret scanning.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook needs a database password. What is the recommended way to provide it?</b><br>
A. Hard-code it in the notebook and restrict notebook permissions<br>
B. Store it in a secret scope and read it with <code>dbutils.secrets.get</code><br>
C. Pass it as a widget default value<br>
D. Store it in a Delta table<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. What is displayed when a notebook prints a value obtained with <code>dbutils.secrets.get</code>?</b><br>
A. The plain value<br>
B. <code>[REDACTED]</code><br>
C. An encrypted string<br>
D. An error<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a scope <code>practice</code> and two secrets, <code>api_key</code> and <code>api_url</code>, with the SDK (use dummy values)</li><li>Build a dictionary of request settings from both secrets and print it (check the redaction)</li><li>List the keys of the scope, then delete the scope</li><li>Write the function <code>get_secret(scope, key)</code> that falls back to an environment variable when <code>dbutils</code> isn't available, for unit tests</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Scope and two dummy secrets
w = WorkspaceClient()
try:
    w.secrets.create_scope(scope="practice")
except Exception:
    pass                                         # already exists
w.secrets.put_secret(scope="practice", key="api_key", string_value="dummy-" + pysecrets.token_hex(8))
w.secrets.put_secret(scope="practice", key="api_url", string_value="https://api.example.com")

# 2. Settings from secrets (values redacted in output)
settings = {"url": dbutils.secrets.get("practice", "api_url"), "key": dbutils.secrets.get("practice", "api_key")}
print(settings)

# 3. Keys, then cleanup
print([m.key for m in dbutils.secrets.list("practice")])
for k in ["api_key", "api_url"]:
    w.secrets.delete_secret(scope="practice", key=k)
w.secrets.delete_scope(scope="practice")

# 4. See get_secret() in the Under the hood section: dbutils on Databricks, environment variables elsewhere.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Never put credentials in notebooks, files or Git. Rotate anything that was exposed</li><li>Secret scopes hold secrets: Databricks-backed or Azure Key Vault-backed</li><li>Create secrets outside notebooks (CLI, SDK, Terraform). Read with <code>dbutils.secrets.get</code></li><li>Output shows <code>[REDACTED]</code>. ACLs (<code>READ</code>, <code>WRITE</code>, <code>MANAGE</code>) control real access</li><li>Prefer UC storage credentials and connections for storage and databases</li><li><code>{{secrets/scope/key}}</code> references secrets in cluster and job configuration</li></ul>
</div>

| Task | Code |
|---|---|
| Create scope (CLI) | `databricks secrets create-scope my_scope` |
| Put secret (CLI) | `databricks secrets put-secret my_scope my_key` |
| Create (SDK) | `w.secrets.create_scope(scope=...)`, `w.secrets.put_secret(...)` |
| Read | `dbutils.secrets.get(scope="my_scope", key="my_key")` |
| List | `dbutils.secrets.listScopes()`, `dbutils.secrets.list("my_scope")` |
| Config reference | `{{secrets/my_scope/my_key}}` |

## Git commit

```
git add 04_databricks/05_secrets_and_scopes.ipynb
git commit -m "add 04_05 secrets and scopes notebook"
```